# Factbird BYOM — Quickstart

Two operations, one notebook:

1. **Deploy** a compiled model to an edge device (drops the artifact into the BYOM S3 bucket; the Factbird backend creates the IoT job).
2. **View** the device's KVS stream via a signed HLS or DASH URL.

### Prerequisites

- Python 3.11+
- Access keys for the `byom-uploader` IAM user (Factbird rotates these ~yearly)
- A device that is already provisioned with Factbird (you have a `device-id` / `stream-id`)

```bash
pip install -e .
export AWS_ACCESS_KEY_ID=...
export AWS_SECRET_ACCESS_KEY=...
export AWS_REGION=eu-west-1
export AWS_ACCOUNT_ID=...   # given to you by Factbird with the access keys
```

The BYOM S3 bucket name is derived by convention — `{account_id}-{region}-byom` — so the SDK builds it from the account ID and region and you never type it out.

In [ ]:
import os

import boto3

from factbird_byom import ClientConfig, FactbirdByomClient

client = FactbirdByomClient(
    session=boto3.Session(),
    config=ClientConfig(
        region=os.environ["AWS_REGION"],
        account_id=os.environ["AWS_ACCOUNT_ID"],
    ),
)

DEVICE_ID = "edge-01"  # replace with your device-id (IoT thing-id)

## 1. Deploy a model

`deploy()` uploads the file to `s3://{bucket}/{device_id}/{utc_epoch}/{model_name}.hef`.
The `device_id` is passed through as-is; it is **not** verified client-side — verification happens server-side when the backend reacts to the S3 event.

In [ ]:
result = client.deployment.deploy("./model.hef", device_id=DEVICE_ID)

print(f"uploaded: s3://{result.bucket}/{result.key}")
print(f"size:     {result.size} bytes")
print(f"etag:     {result.etag}")

## 2. View the live stream

The SDK mints a signed HLS or DASH URL. Paste it into any HLS/DASH-capable player:

- **Safari** (macOS / iOS): paste URL in address bar — plays natively.
- **VLC**: `vlc <url>` or File → Open Network.
- **ffplay**: `ffplay <url>`.
- **QuickTime**: File → Open Location (HLS only).

The default `expires_in=300` gives a 5-minute window. Minimum is 300s, maximum 43200s (12h).

In [ ]:
with client.streams.open(DEVICE_ID) as kvs:
    hls = kvs.hls_url(expires_in=300)
    dash = kvs.dash_url(expires_in=300)

print("HLS: ", hls)
print("DASH:", dash)

### Opening the URL from Python

You can also launch the platform's default viewer directly:

In [ ]:
import subprocess
import sys

if sys.platform == "darwin":
    subprocess.Popen(["open", hls])
elif sys.platform.startswith("linux"):
    subprocess.Popen(["xdg-open", hls])
else:
    print("paste the URL into your player of choice")

### Or from the shell, no Python

The CLI entry point does the same thing in one line:

```bash
factbird-byom view edge-01
factbird-byom view edge-01 --dash --expires 600
factbird-byom view edge-01 --no-open  # just prints the URL
```